# 4. SQL / Python Tool 최소 권한

이 노트북은 Week04 [`4. SQLite3 with QuerySQLDatabaseTool.ipynb`](../../../week04_타당성검토_에이전트기획/day04_4일차/AI%20에이전트%20예제/4.%20SQLite3%20with%20QuerySQLDatabaseTool.ipynb)의 `search_employee_by_name` Tool과 [`2. Python Tools.ipynb`](../../../week04_타당성검토_에이전트기획/day04_4일차/AI%20에이전트%20예제/2.%20Python%20Tools.ipynb)의 `python_tool`을 그대로 재사용해서, Tool **내부**에서 최소 권한을 강제하지 않으면 어떤 일이 벌어지는지 확인합니다.

이번 노트북은 3번 노트북과 달리 **Agent/LLM을 거치지 않고 Tool 함수를 직접 호출**합니다. 왜냐하면 실제 공격에서는 "LLM이 이런 입력을 순순히 만들어줄까?"가 아니라, **"이런 입력이 들어왔을 때 Tool이 안전한가?"**가 핵심이기 때문입니다. LLM이 실수로, 혹은 Prompt Injection에 당해서, 혹은 악의적인 사용자가 애매한 표현으로 유도해서 — 어떤 경로로든 위험한 인자가 Tool에 들어올 수 있습니다. **Tool 자체가 방어선이 되어야 합니다.**

- **A 코드**: Week04에서 만든 SQL/Python Tool을 그대로 실행 → 위험한 입력을 직접 넣어서 문제 확인
- **B 코드**: 같은 Tool에 "금지 키워드 검사"만 추가 → 동일한 위험한 입력이 차단되는 것을 확인

## 왜 `sql_db_query_checker`만으로는 부족한가요?

`SQLDatabaseToolkit`에는 `sql_db_query_checker`라는 Tool이 있어서, 실행 전에 **LLM에게 한 번 더** "이 쿼리가 맞는지" 물어보게 할 수 있습니다. 하지만:

- 이건 여전히 **모델의 판단에 의존하는 방식**입니다. 모델이 위험한 쿼리를 위험하다고 못 알아채면 그대로 통과됩니다.
- System Prompt에 "DML(INSERT/UPDATE/DELETE/DROP)은 절대 하지 마세요"라고 적어놓아도, 이는 **강제가 아니라 권고**일 뿐입니다 — 모델이 프롬프트를 우회하거나 무시하면 그대로 실행됩니다.

이번 노트북은 모델을 더 똑똑하게 만드는 대신, **Tool 코드 자체가 위험한 입력을 걸러내도록** 만듭니다. 이 방식은 "모델이 무엇을 하려고 했는지"와 무관하게 항상 동일하게 동작하는 결정적(deterministic)인 방어선입니다.

Python 실행 Tool도 마찬가지입니다 — `timeout`만으로는 "무한 루프를 5초 안에 멈추는 것"만 막을 뿐, 파일을 읽거나 환경변수를 훔쳐가거나 네트워크에 접속하는 코드는 5초 안에 얼마든지 실행될 수 있습니다.

## Part 1. SQL Tool — 최소 권한 없는 문자열 조합의 위험

### 실습 준비: 강의와 동일한 샘플 DB 생성

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sqlite3

DB_PATH = os.getenv("SECURITY_DEMO_SQLITE_PATH", "security_demo.sqlite")

def create_sample_database():
    """학습용 샘플 데이터베이스를 생성합니다. (강의 노트북과 동일)"""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS employees (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            department TEXT NOT NULL,
            salary INTEGER NOT NULL,
            hire_date TEXT NOT NULL
        )
    ''')

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS departments (
            id INTEGER PRIMARY KEY,
            dept_name TEXT NOT NULL,
            manager TEXT NOT NULL,
            budget INTEGER NOT NULL
        )
    ''')

    employees_data = [
        (1, "김철수", "IT", 5000, "2020-01-15"),
        (2, "이영희", "HR", 4500, "2019-03-20"),
        (3, "박민수", "IT", 5500, "2021-07-10"),
        (4, "최지영", "Sales", 4000, "2020-09-05"),
        (5, "정수현", "Marketing", 4200, "2022-01-25"),
    ]
    cursor.executemany("INSERT OR REPLACE INTO employees VALUES (?, ?, ?, ?, ?)", employees_data)

    departments_data = [
        (1, "IT", "김철수", 100000),
        (2, "HR", "이영희", 50000),
        (3, "Sales", "최지영", 80000),
        (4, "Marketing", "정수현", 60000),
    ]
    cursor.executemany("INSERT OR REPLACE INTO departments VALUES (?, ?, ?, ?)", departments_data)

    conn.commit()
    conn.close()
    print("샘플 데이터베이스가 생성되었습니다!")

create_sample_database()

샘플 데이터베이스가 생성되었습니다!


In [3]:
from langchain_community.utilities import SQLDatabase
from langchain_community.tools.sql_database.tool import QuerySQLDatabaseTool

db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH}")
sql_tool = QuerySQLDatabaseTool(db=db)

In [4]:
from langchain.tools import tool
import ast


@tool
def search_employee_by_name(name: str) -> str:
    """이름으로 직원을 검색합니다."""
    try:
        query = f"""
        SELECT e.name, e.department, e.salary, e.hire_date, d.manager, d.budget
        FROM employees e
        LEFT JOIN departments d ON e.department = d.dept_name
        WHERE e.name LIKE '%{name}%'
        """
        result = sql_tool.invoke(query)
        if result:
            return f"'{name}' 검색 결과:\n{ast.literal_eval(result)}"
        else:
            return f"'{name}'과 일치하는 직원을 찾을 수 없습니다."
    except Exception as e:
        return f"직원 검색 중 오류가 발생했습니다: {str(e)}"

### [A] 실행: 정상적인 검색은 잘 동작함

In [5]:
print(search_employee_by_name.invoke({"name": "김철수"}))

'김철수' 검색 결과:
[('김철수', 'IT', 5000, '2020-01-15', '김철수', 100000)]


### [A] 실행: SQL Injection — "이름 한 명 검색"용 Tool로 전체 직원 정보를 통째로 덤프

`name` 파라미터가 그대로 f-string에 꽂혀 들어가기 때문에, SQL 문법을 포함한 문자열을 넣으면 **쿼리의 의미 자체를 바꿔버릴 수 있습니다.**

In [6]:
malicious_name = "존재하지않는이름' OR '1'='1' -- "
print(search_employee_by_name.invoke({"name": malicious_name}))

'존재하지않는이름' OR '1'='1' -- ' 검색 결과:
[('김철수', 'IT', 5000, '2020-01-15', '김철수', 100000), ('이영희', 'HR', 4500, '2019-03-20', '이영희', 50000), ('박민수', 'IT', 5500, '2021-07-10', '김철수', 100000), ('최지영', 'Sales', 4000, '2020-09-05', '최지영', 80000), ('정수현', 'Marketing', 4200, '2022-01-25', '정수현', 60000)]


**문제 확인**: 분명 "존재하지않는이름"으로 검색했는데, **직원 5명 전체의 이름·부서·급여 정보가 전부** 출력되었을 것입니다. 실제 쿼리는 다음과 같이 조합되었기 때문입니다:

```sql
WHERE e.name LIKE '%존재하지않는이름' OR '1'='1' -- %'
```

`--`는 SQL 주석 시작 기호라서 그 뒤(원래 있어야 할 `%'`)는 통째로 무시되고, `'1'='1'`은 항상 참이므로 `WHERE` 절 전체가 사실상 무력화되어 테이블의 모든 행이 반환됩니다. 이것이 바로 **SQL Injection**입니다 — "직원 한 명만 조회해야 하는" Tool이 "테이블 전체를 덤프하는" Tool로 둔갑했습니다.

## [B] 가드레일 추가: 금지 키워드/문자 검사

**`search_employee_by_name`의 쿼리 조합 코드는 한 글자도 바꾸지 않고**, 그 앞에 입력값 검사만 추가합니다. (참고: 실무에서는 파라미터 바인딩(prepared statement)이 근본적인 해법이지만, 이번 노트북은 README에 명시된 대로 **가장 기초적인 "금지 키워드 포함 여부 검사"** 방식을 실습합니다.)

In [7]:
SQL_INJECTION_MARKERS = ["'", '"', "--", ";", " or ", " union ", "drop", "delete", "update", "insert", "attach", "pragma"]


def is_suspicious_sql_input(value: str) -> bool:
    """검색어에 SQL 인젝션에 흔히 쓰이는 문자/키워드가 포함되어 있는지 검사합니다."""
    lowered = f" {value.lower()} "
    return any(marker in lowered for marker in SQL_INJECTION_MARKERS)


@tool
def search_employee_by_name_guarded(name: str) -> str:
    """(가드레일 적용) 이름으로 직원을 검색합니다."""
    if is_suspicious_sql_input(name):
        return "허용되지 않는 문자가 포함된 검색어입니다. 이름만 입력해 주세요."
    return search_employee_by_name.invoke({"name": name})  # 기존 강의 Tool을 그대로 재사용

### [B] 실행: 정상 검색은 그대로 동작

In [8]:
print(search_employee_by_name_guarded.invoke({"name": "김철수"}))

'김철수' 검색 결과:
[('김철수', 'IT', 5000, '2020-01-15', '김철수', 100000)]


### [B] 실행: 동일한 SQL Injection 페이로드 → 차단됨

In [9]:
print(search_employee_by_name_guarded.invoke({"name": malicious_name}))

허용되지 않는 문자가 포함된 검색어입니다. 이름만 입력해 주세요.


**문제 확인**: 이번에는 전체 덤프 대신 차단 메시지가 출력됩니다. `' OR '1'='1` 안의 작은따옴표(`'`)와 `" or "` 키워드가 감지되었기 때문입니다.

## Part 2. Python 실행 Tool — 샌드박스 없는 코드 실행의 위험

### 실습 준비: 강의와 동일한 Python 실행 Tool

> Windows + 한글 환경에서 자식 프로세스의 출력이 깨지지 않도록 `encoding="utf-8"`만 추가했습니다 (보안과 무관한 인코딩 호환성 수정이며, 위험한 입력을 막는 로직은 전혀 없다는 점은 강의 원본과 동일합니다).

In [10]:
import subprocess
import sys


@tool
def python_tool(code: str) -> str:
    """Python 코드를 실행합니다."""
    try:
        python_exec = "python3" if sys.platform != "win32" else "python"
        result = subprocess.run(
            [python_exec, "-c", code],
            capture_output=True,
            text=True,
            timeout=5,
            encoding="utf-8",
            errors="replace",
        )
        return result.stdout if result.stdout else result.stderr
    except Exception as e:
        return f"오류: {str(e)}"

### [A] 실행: 정상적인 계산은 잘 동작함

In [11]:
print(python_tool.invoke({"code": "print(2**10)"}))

1024



### [A] 실행: 파일 시스템 접근 — 이 서버에 어떤 파일이 있는지 그대로 노출

In [12]:
print(python_tool.invoke({"code": "import os; print(os.listdir('.'))"}))

['.env', '.env.example', '.python-version', '.venv', '1. Agent 보안 개요와 Prompt Injection.ipynb', '2. 입력 출력 Guardrail.ipynb', '3. Tool Guardrails 권한 기반 실행.ipynb', '4. SQL Python Tool 최소 권한.ipynb', 'pyproject.toml', 'README.md', 'security_demo.sqlite', 'uv.lock']



**문제 확인**: `timeout=5`는 "5초 안에 끝나지 않으면 죽인다"만 보장할 뿐, **그 5초 동안 무엇을 하든 막지 않습니다.** 위 코드는 서버의 현재 디렉터리 파일 목록(`.env` 파일 포함 여부 등)을 그대로 노출시킵니다. 실제 공격이라면 여기서 `open('.env').read()`까지 실행해서 `GROQ_API_KEY` 같은 진짜 비밀값을 그대로 훔쳐갈 수 있습니다 — 이 노트북에서는 안전을 위해 실행하지 않지만, 코드 구조상 **막을 방법이 전혀 없다는 것**이 핵심입니다.

### [A] 실행: 환경변수(비밀값) 탈취

`DEMO_API_KEY`는 이 실습을 위해 `.env`에 넣어둔 **가짜** 비밀값입니다 (`.env.example` 참고). 실제 서비스라면 이 자리에 진짜 API 키나 DB 비밀번호가 들어있을 수 있습니다.

In [13]:
print(python_tool.invoke({"code": "import os; print('탈취된 내부 키:', os.environ.get('DEMO_API_KEY'))"}))

탈취된 내부 키: demo-lecture-only



**문제 확인**: `python_tool`이 실행하는 프로세스는 우리 애플리케이션과 같은 환경변수를 그대로 물려받습니다. `os`, `subprocess`, `open` 등 아무 모듈이나 자유롭게 `import`할 수 있기 때문에, 파일이든 환경변수든 네트워크든 **이 프로세스가 접근할 수 있는 모든 것에 접근할 수 있습니다.**

## [B] 가드레일 추가: 금지 키워드 검사 (+ 기존 timeout 유지)

In [14]:
FORBIDDEN_PY_KEYWORDS = [
    "import os", "import subprocess", "import sys", "import shutil", "import socket",
    "os.", "subprocess.", "socket.", "open(", "__import__", "eval(", "exec(",
]


def is_dangerous_code(code_str: str) -> bool:
    """os/subprocess/open 등 위험한 모듈·함수 사용을 문자열 포함 여부로 검사합니다."""
    lowered = code_str.lower()
    return any(keyword.lower() in lowered for keyword in FORBIDDEN_PY_KEYWORDS)


@tool
def python_tool_guarded(code: str) -> str:
    """(가드레일 적용) Python 코드를 실행합니다."""
    if is_dangerous_code(code):
        return "허용되지 않는 코드입니다 (os/subprocess/open 등 파일·시스템 접근은 금지되어 있습니다)."
    try:
        python_exec = "python3" if sys.platform != "win32" else "python"
        result = subprocess.run(
            [python_exec, "-c", code],
            capture_output=True,
            text=True,
            timeout=5,  # 기존 timeout도 그대로 유지 (무한 루프 방지)
            encoding="utf-8",
            errors="replace",
        )
        return result.stdout if result.stdout else result.stderr
    except Exception as e:
        return f"오류: {str(e)}"

### [B] 실행: 정상 계산은 그대로 동작

In [15]:
print(python_tool_guarded.invoke({"code": "print(2**10)"}))

1024



### [B] 실행: 동일한 파일 시스템 접근 / 환경변수 탈취 시도 → 차단됨

In [16]:
print(python_tool_guarded.invoke({"code": "import os; print(os.listdir('.'))"}))
print(python_tool_guarded.invoke({"code": "import os; print('탈취된 내부 키:', os.environ.get('DEMO_API_KEY'))"}))

허용되지 않는 코드입니다 (os/subprocess/open 등 파일·시스템 접근은 금지되어 있습니다).
허용되지 않는 코드입니다 (os/subprocess/open 등 파일·시스템 접근은 금지되어 있습니다).


## 정리

| | A (가드레일 없음) | B (가드레일 추가) |
| --- | --- | --- |
| SQL Tool 쿼리 조합 코드 | 강의 원본 그대로 (f-string) | 완전히 동일 (수정 없음) |
| `' OR '1'='1` 검색 | 전체 테이블 덤프 | 의심 문자 감지되어 차단 |
| Python 실행 Tool 코드 | 강의 원본 그대로 (subprocess + timeout) | 완전히 동일 (수정 없음) |
| `import os; ...` 코드 | 파일 목록/환경변수 그대로 노출 | 금지 키워드 감지되어 차단 |

**기억할 점**
- 이번 노트북의 방어는 **키워드 블록리스트**입니다. 빠르고 구현하기 쉽지만, `impo` + `rt os`처럼 문자열을 쪼개거나 인코딩해서 우회하는 시도까지 막지는 못합니다. 실무에서는 SQL은 **파라미터 바인딩(prepared statement)**, Python은 **진짜 샌드박스(별도 컨테이너/프로세스 격리, 화이트리스트 기반 AST 검사 등)**로 한 단계 더 강화해야 합니다.
- 그래도 "아무 방어도 없는 것"과 "기본적인 키워드 검사라도 있는 것"의 차이는 이번 실습에서 본 것처럼 매우 큽니다 — **최소한의 방어선도 없는 Tool을 그대로 배포하지 마세요.**